In [2]:
# ==============================================================================
#  PIPELINE ML — DÉTECTION VÉRIFICATION FACTURES
#  Transformation : Système à règles → Machine Learning supervisé
# ------------------------------------------------------------------------------
#  Contexte :
#    Les colonnes du dataset sont le résultat de règles métier appliquées sur
#    des factures médicales (complétude, validation, contrôles, existence).
#    On cherche à prédire : status_verification  (validee / a_corriger)
#
#  Stratégie :
#    1. Feature engineering : créer des scores synthétiques par groupe de règles
#    2. Split 60/20/20 : train / validation (calibration seuil) / test (évaluation)
#    3. Optimisation du seuil de décision sur le set de validation
#    4. Évaluation finale sur le test set indépendant
#    5. Visualisations complètes : learning curves, matrices de confusion,
#       courbes ROC/PR, comparaison inter-modèles
#
#  Cibles réalistes (plafond théorique des données = 64.2%) :
#    AUC       : 0.68–0.71
#    Recall    : 0.72–0.80
#    F1        : 0.68–0.74
#
#  Modèles    : GradientBoosting | RandomForest | KNN | LogisticRegression
#  Auteur     : Pipeline ML Expert
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, learning_curve
)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

# ──────────────────────────────────────────────────────────────────────────────
# 0. CONFIGURATION GLOBALE
# ──────────────────────────────────────────────────────────────────────────────

RANDOM_STATE = 42
DATA_PATH    = 'data/dfforml2.csv'

# Couleurs dédiées à chaque modèle (cohérence sur tous les graphes)
MODEL_COLORS = {
    'GradientBoosting' : '#2196F3',   # Bleu
    'RandomForest'     : '#4CAF50',   # Vert
    'KNN'              : '#FF9800',   # Orange
    'LogisticReg'      : '#9C27B0',   # Violet
}

# Style général des graphes
plt.rcParams.update({
    'figure.dpi'     : 130,
    'font.size'      : 10,
    'axes.titlesize' : 12,
    'axes.titleweight': 'bold',
    'axes.grid'      : True,
    'grid.alpha'     : 0.3,
})

print("=" * 70)
print("   PIPELINE ML — VÉRIFICATION FACTURES MÉDICALES")
print("=" * 70)


# ──────────────────────────────────────────────────────────────────────────────
# 1. CHARGEMENT DES DONNÉES
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 1/7] Chargement des données")

df = pd.read_csv(DATA_PATH)

# Encodage de la variable cible
# validee = 0  (classe négative)
# a_corriger = 1  (classe positive — ce qu'on veut détecter)
y = df['status_verification'].map({'validee': 0, 'a_corriger': 1})

n_total     = len(df)
n_corriger  = y.sum()
n_validee   = (y == 0).sum()

print(f"  ➤ Lignes totales       : {n_total}")
print(f"  ➤ Classe 'validee'     : {n_validee} ({n_validee/n_total*100:.1f}%)")
print(f"  ➤ Classe 'a_corriger'  : {n_corriger} ({n_corriger/n_total*100:.1f}%)")
print(f"  ➤ Déséquilibre         : ratio = {n_corriger/n_validee:.2f}")
print(f"  ➤ Plafond théorique    : 64.2% (limité par le bruit irréductible des données)")


# ──────────────────────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────
# Principe : les 37 colonnes sont des flags binaires issus de 4 groupes de règles
# métier. La majorité sont constantes (>97% même valeur) et donc inutiles seules.
# On crée des SCORES SYNTHÉTIQUES par groupe pour capturer les interactions
# que les features binaires individuelles ne peuvent pas exprimer.
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 2/7] Feature Engineering — Création des scores synthétiques")

# ── Groupe 1 : Complétude des champs obligatoires ──
# Règles qui vérifient si les champs clés sont renseignés (0 = manquant)
COMPLETUDE = [
    'nom_patient_is_filled',       # Nom du patient présent
    'distance_village_is_filled',  # Distance village renseignée
    'age_patient_is_filled',       # Âge patient renseigné
    'registre_number_is_filled',   # Numéro de registre présent
    'id_prescripteur_is_filled',   # Identifiant prescripteur
    'id_gerant_is_filled',         # Identifiant gérant
    'sex_is_filled',               # Sexe renseigné
    'consultation_type_is_filled', # Type de consultation renseigné
    'type_prestation_is_filled',   # Type de prestation renseigné
    'visit_date_is_filled',        # Date de visite renseignée
]

# ── Groupe 2 : Validation des valeurs ──
# Règles qui vérifient la validité logique des données saisies
VALIDATION = [
    'age_patient_is_valid',    # Âge dans les bornes acceptables
    'sex_is_valid',            # Sexe parmi les valeurs autorisées
    'date_entree_is_valid',    # Date d'entrée logiquement valide
    'date_sortie_is_valid',    # Date de sortie logiquement valide
    'visit_date_is_valid',     # Date de visite valide
    'created_at_is_valid',     # Date de création valide
]

# ── Groupe 3 : Contrôles métier (règles d'incohérence) ──
# Règles qui détectent des incohérences entre plusieurs champs
CONTROLES = [
    'verifierIncoherenceDateEntree',         # Incohérence sur date entrée
    'verifierIncoherenceDateSortie',         # Incohérence sur date sortie
    'verifierChevauchementHospitalisation',  # Chevauchement hospitalisations
    'verifierIncoherenceSexe',               # Incohérence sexe/prestation
    'verifierMontantEvacuation',             # Montant évacuation anormal
    'verifierHospitalisationEtEvacuation',   # Double hospitalisation+évacuation
    'verifierEvacuationIncoherence',         # Évacuation incohérente
    'verifierHopitalisation_PF_Ambulatoires',# Hospit. + PF ambulatoire
    'verifierPrestationEnfant',              # Prestation enfant incohérente
]

# ── Groupe 4 : Existence des champs de valeurs ──
# Règles qui vérifient la présence de champs numériques (quantités, coûts)
EXISTENCE = [
    'quantite_total_prod_exists',       # Quantité produits présente
    'quantite_total_act_exists',        # Quantité actes présente
    'quantite_total_ex_exists',         # Quantité examens présente
    'cout_total_prod_exists',           # Coût produits présent
    'cout_total_act_exists',            # Coût actes présent
    'cout_total_ex_exists',             # Coût examens présent
    'cout_mise_en_observation_exists',  # Coût observation présent
    'cout_evacuation_exists',           # Coût évacuation présent
    'nbre_jours_exists',                # Nombre de jours présent
]

# ── Construction des features synthétiques ──
df2 = df.copy()

# [FE-1] Nombre de champs non renseignés (incompletude)
# Plus ce score est élevé, plus la facture est incomplète
df2['score_incompletude'] = df[COMPLETUDE].apply(
    lambda row: (row == 0).sum(), axis=1
)

# [FE-2] Nombre de valeurs invalides
# Capture les anomalies de validation (dates impossibles, etc.)
df2['score_invalidite'] = df[VALIDATION].apply(
    lambda row: (row == 0).sum(), axis=1
)

# [FE-3] Nombre d'incohérences détectées par les contrôles métier
# Ce score est le signal le plus fort car il encode des règles expertes
df2['score_incoherences'] = df[CONTROLES].sum(axis=1)

# [FE-4] Nombre de champs de valeurs absents
# Factures sans quantités ni coûts = suspectes
df2['score_absence_valeurs'] = df[EXISTENCE].apply(
    lambda row: (row == 0).sum(), axis=1
)

# [FE-5] Score global d'anomalie — combinaison pondérée des 4 groupes
# Pondération : contrôles métier × 2 (experts), validation × 1.5, etc.
df2['score_global_anomalie'] = (
    df2['score_incompletude']    * 1.0 +
    df2['score_invalidite']      * 1.5 +   # Valeurs invalides = signal modéré
    df2['score_incoherences']    * 2.0 +   # Règles expertes = signal fort
    df2['score_absence_valeurs'] * 0.5     # Absence valeurs = signal faible
)

# [FE-6] Interaction consultation × prestation
# Certaines combinaisons sont quasi-déterministes (ex: type_presta=21 → 90% a_corriger)
df2['consult_presta_combo'] = (
    df['consultation_type'].fillna(0).astype(int) * 100 +
    df['type_prestation'].fillna(0).astype(int)
)

# [FE-7] Flag binaire : au moins une incohérence détectée
# Simplifie score_incoherences en signal binaire fort
df2['has_incoherence'] = (df2['score_incoherences'] > 0).astype(int)

# [FE-8] Interaction : examens présents ET incohérence détectée
# Combinaison spécifique liée à "quantité anormale d'acte"
df2['examen_avec_anomalie'] = (
    (df['quantite_total_ex_exists'] == 1) &
    (df2['score_incoherences'] > 0)
).astype(int)

# [FE-9] Taux d'incomplétude (proportion de champs manquants)
# Normalise score_incompletude entre 0 et 1
df2['taux_incompletude'] = df2['score_incompletude'] / len(COMPLETUDE)

# [FE-10] Profil de risque discrétisé (0=faible, 1=moyen, 2=élevé)
# Transforme le score continu en catégorie ordinal interprétable
df2['profil_risque'] = pd.cut(
    df2['score_global_anomalie'],
    bins=[-np.inf, 1, 3, np.inf],
    labels=[0, 1, 2]
).astype(int)

# [FE-11] Flag consultation type 5 (80% de a_corriger dans les données)
# Encode directement la connaissance métier observée
df2['consult5_flag'] = (df['consultation_type'] == 5).astype(int)

# [FE-12] Flag prestation à haut risque (certains codes → quasi 100% a_corriger)
# Basé sur l'analyse des croisements type_prestation × status
PRESTATIONS_RISQUE = [21, 22, 23, 25, 27, 28, 29, 30, 9, 12, 16]
df2['presta_risque'] = df['type_prestation'].apply(
    lambda x: 1 if x in PRESTATIONS_RISQUE else 0
)

# [FE-13] Produit : hospitalisation_PF × examens
# Interaction multiplicative entre deux features utiles
df2['hosp_pf_examen'] = (
    df['verifierHopitalisation_PF_Ambulatoires'] *
    df['quantite_total_ex_exists']
)

# ── Liste finale des features utilisées ──
FEATURES_ORIGINALES = [
    'consultation_type',                     # Catégorielle (5 valeurs)
    'type_prestation',                       # Catégorielle (20 valeurs)
    'date_entree_is_valid',                  # Binaire (10% variance)
    'date_sortie_is_valid',                  # Binaire (10% variance)
    'verifierHopitalisation_PF_Ambulatoires',# Binaire (30% variance)
    'quantite_total_ex_exists',              # Binaire (16% variance)
]

FEATURES_SYNTHETIQUES = [
    'score_incompletude',     # [FE-1]
    'score_invalidite',       # [FE-2]
    'score_incoherences',     # [FE-3]
    'score_absence_valeurs',  # [FE-4]
    'score_global_anomalie',  # [FE-5]
    'consult_presta_combo',   # [FE-6]
    'has_incoherence',        # [FE-7]
    'examen_avec_anomalie',   # [FE-8]
    'taux_incompletude',      # [FE-9]
    'profil_risque',          # [FE-10]
    'consult5_flag',          # [FE-11]
    'presta_risque',          # [FE-12]
    'hosp_pf_examen',         # [FE-13]
]

ALL_FEATURES = FEATURES_ORIGINALES + FEATURES_SYNTHETIQUES

X = df2[ALL_FEATURES].fillna(-1)   # -1 pour les valeurs manquantes (type_observation=NaN)

print(f"  ➤ Features originales utiles : {len(FEATURES_ORIGINALES)}")
print(f"  ➤ Features synthétiques      : {len(FEATURES_SYNTHETIQUES)}")
print(f"  ➤ Total features             : {len(ALL_FEATURES)}")

# Corrélations avec la target pour valider l'intérêt des nouvelles features
print("\n  Corrélations features synthétiques vs target :")
for feat in FEATURES_SYNTHETIQUES:
    corr = df2[feat].corr(y)
    bar  = '█' * int(abs(corr) * 30)
    print(f"    {feat:<30} {corr:+.4f}  {bar}")


# ──────────────────────────────────────────────────────────────────────────────
# 3. SPLIT TRAIN / VALIDATION / TEST
# ──────────────────────────────────────────────────────────────────────────────
# On utilise un split en 3 parties pour éviter le biais d'optimisation du seuil :
#   - TRAIN (60%)      : entraînement des modèles
#   - VALIDATION (20%) : calibration du seuil de décision
#   - TEST (20%)       : évaluation finale indépendante
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 3/7] Split Train / Validation / Test (60/20/20)")

# Split initial : 80% train+val / 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = RANDOM_STATE,
    stratify     = y            # Préserve les proportions de classes
)

# Split secondaire : 75% train / 25% val (soit 60/20 du total)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size    = 0.25,
    random_state = RANDOM_STATE,
    stratify     = y_trainval
)

print(f"  ➤ Train      : {len(X_train):4d} lignes  "
      f"(a_corriger={y_train.sum()}, {y_train.mean()*100:.1f}%)")
print(f"  ➤ Validation : {len(X_val):4d} lignes  "
      f"(a_corriger={y_val.sum()}, {y_val.mean()*100:.1f}%)")
print(f"  ➤ Test       : {len(X_test):4d} lignes  "
      f"(a_corriger={y_test.sum()}, {y_test.mean()*100:.1f}%)")


# ──────────────────────────────────────────────────────────────────────────────
# 4. DÉFINITION DES MODÈLES
# ──────────────────────────────────────────────────────────────────────────────
# Chaque modèle est encapsulé dans un Pipeline sklearn :
#   StandardScaler → normalise les features continues
#   Classifier     → apprend les patterns
# Le scaler est nécessaire pour KNN et LogisticReg (sensibles à l'échelle).
# Pour GBM et RF il n'a pas d'effet mais uniformise le code.
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 4/7] Définition et entraînement des modèles")

models = {

    # ── GradientBoosting ──────────────────────────────────────────────────────
    # Boosting séquentiel : chaque arbre corrige les erreurs du précédent.
    # Très performant sur données tabulaires. Sensible au surapprentissage
    # → learning_rate faible + subsample pour régulariser.
    'GradientBoosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators     = 400,   # Nb d'estimateurs (arbres)
            learning_rate    = 0.03,  # Faible → meilleure généralisation
            max_depth        = 5,     # Profondeur max par arbre
            min_samples_leaf = 8,     # Feuilles avec ≥8 exemples
            subsample        = 0.8,   # 80% des données par arbre (stochastic GB)
            random_state     = RANDOM_STATE
        ))
    ]),

    # ── RandomForest ─────────────────────────────────────────────────────────
    # Ensemble d'arbres indépendants (bagging). Robuste et parallélisable.
    # class_weight='balanced' : compense le déséquilibre 58/42 automatiquement.
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            n_estimators     = 400,        # Nb d'arbres dans la forêt
            max_depth        = 10,         # Profondeur maximale
            min_samples_leaf = 5,          # Régularisation
            max_features     = 'sqrt',     # Sélection aléatoire des features
            class_weight     = 'balanced', # Pondère les classes pour recall
            random_state     = RANDOM_STATE,
            n_jobs           = -1          # Parallélisation sur tous les cœurs
        ))
    ]),

    # ── KNN (K plus proches voisins) ─────────────────────────────────────────
    # Classifie en fonction de la majorité parmi les k voisins les plus proches.
    # Très sensible à l'échelle des features → StandardScaler OBLIGATOIRE.
    # weights='distance' : les voisins proches influencent plus la décision.
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(
            n_neighbors = 15,         # k=15 : équilibre biais/variance
            weights     = 'distance', # Pondération par inverse de la distance
            metric      = 'euclidean',
            n_jobs      = -1
        ))
    ]),

    # ── LogisticRegression ───────────────────────────────────────────────────
    # Modèle linéaire probabiliste, très interprétable.
    # class_weight='balanced' : compense le déséquilibre.
    # C=1.0 : régularisation L2 standard.
    'LogisticReg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            C            = 1.0,         # Inverse de la régularisation
            max_iter     = 2000,        # Convergence assurée
            class_weight = 'balanced',  # Pondération automatique
            solver       = 'lbfgs',
            random_state = RANDOM_STATE
        ))
    ]),
}

# ── Entraînement ──────────────────────────────────────────────────────────────
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name:<22} entraîné")


# ──────────────────────────────────────────────────────────────────────────────
# 5. OPTIMISATION DU SEUIL DE DÉCISION
# ──────────────────────────────────────────────────────────────────────────────
# Par défaut, sklearn classe en "a_corriger" si P(a_corriger) ≥ 0.5.
# Mais le seuil optimal dépend du coût métier (manquer une erreur est coûteux).
# Stratégie : chercher le seuil qui maximise le F1 tout en garantissant
# un Recall ≥ 0.72 — calibré sur le SET DE VALIDATION, appliqué au TEST.
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 5/7] Optimisation du seuil de décision (sur set de validation)")

def optimize_threshold(y_true, y_proba, recall_min=0.72):
    """
    Cherche le seuil optimal dans [0.20, 0.80] qui :
      1. Garantit Recall ≥ recall_min sur la classe 'a_corriger'
      2. Maximise le F1-score (parmi les seuils satisfaisant la contrainte)

    Si aucun seuil ne satisfait recall_min, retourne le seuil
    qui maximise le F1 sans contrainte.

    Paramètres
    ----------
    y_true     : array-like, labels réels (0/1)
    y_proba    : array-like, probabilités prédites pour la classe 1
    recall_min : float, recall minimum requis (défaut = 0.72)

    Retourne
    --------
    best_t   : float, seuil optimal
    details  : dict, métriques au seuil optimal
    all_res  : list of dict, résultats pour chaque seuil testé
    """
    thresholds = np.arange(0.20, 0.80, 0.01)
    best_t   = 0.50
    best_f1  = 0.0
    all_res  = []

    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        rec   = recall_score(y_true, preds, zero_division=0)
        prec  = precision_score(y_true, preds, zero_division=0)
        f1    = f1_score(y_true, preds, zero_division=0)
        all_res.append({'threshold': t, 'recall': rec,
                        'precision': prec, 'f1': f1})
        # Contrainte : recall ≥ recall_min ET meilleur F1
        if rec >= recall_min and f1 > best_f1:
            best_f1 = f1
            best_t  = t

    # Si aucun seuil ne satisfait la contrainte, prendre le max F1 sans contrainte
    if best_f1 == 0.0:
        best_t = max(all_res, key=lambda x: x['f1'])['threshold']

    best_preds = (y_proba >= best_t).astype(int)
    details = {
        'threshold' : best_t,
        'recall'    : recall_score(y_true, best_preds, zero_division=0),
        'precision' : precision_score(y_true, best_preds, zero_division=0),
        'f1'        : f1_score(y_true, best_preds, zero_division=0),
    }
    return best_t, details, all_res


# Dictionnaire qui stockera toutes les prédictions et métriques
results       = {}          # Métriques finales par modèle
probas_test   = {}          # Probabilités sur test set
optimal_thrs  = {}          # Seuils optimaux
threshold_curves = {}       # Courbes seuil vs métriques (pour graphe)

print(f"\n  {'Modèle':<22} {'Seuil':>6} {'AUC':>7} {'Precision':>10} {'Recall':>8} {'F1':>7}")
print("  " + "─" * 65)

for name, model in models.items():
    # Probabilités sur val et test
    y_proba_val  = model.predict_proba(X_val)[:, 1]
    y_proba_test = model.predict_proba(X_test)[:, 1]

    # Seuil calibré sur le set de VALIDATION
    best_t, val_details, all_res = optimize_threshold(
        y_val, y_proba_val, recall_min=0.72
    )

    # Prédictions finales sur le TEST avec le seuil calibré
    y_pred = (y_proba_test >= best_t).astype(int)

    # Calcul de toutes les métriques sur le TEST
    auc      = roc_auc_score(y_test, y_proba_test)
    acc      = accuracy_score(y_test, y_pred)
    prec     = precision_score(y_test, y_pred, zero_division=0)
    rec      = recall_score(y_test, y_pred, zero_division=0)
    f1       = f1_score(y_test, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results[name] = {
        'threshold'  : best_t,
        'AUC'        : auc,
        'Accuracy'   : acc,
        'Precision'  : prec,
        'Recall'     : rec,
        'F1'         : f1,
        'Specificity': specificity,
        'TP' : tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'y_pred'     : y_pred,
    }
    probas_test[name]      = y_proba_test
    optimal_thrs[name]     = best_t
    threshold_curves[name] = all_res

    print(f"  {name:<22} {best_t:>6.2f} {auc:>7.4f} {prec:>10.4f} {rec:>8.4f} {f1:>7.4f}")


# ──────────────────────────────────────────────────────────────────────────────
# 6. GRAPHES D'APPRENTISSAGE (LEARNING CURVES)
# ──────────────────────────────────────────────────────────────────────────────
# But : vérifier si les modèles sont en sous-apprentissage (underfitting)
# ou sur-apprentissage (overfitting) en fonction de la taille du dataset.
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 6/7] Génération des visualisations...")
print("  → Graphe 1 : Learning Curves")

fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle(
    "Courbes d'Apprentissage par Modèle\n"
    "(Gap train/val faible = bonne généralisation | "
    "Score qui monte = plus de données aiderait)",
    fontsize=13, fontweight='bold', y=1.01
)

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_sizes_pct = np.linspace(0.15, 1.0, 10)   # De 15% à 100% des données train

for ax, (name, model) in zip(axes.flatten(), models.items()):
    color = MODEL_COLORS[name]

    # learning_curve calcule les scores train/val pour différentes tailles
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_trainval, y_trainval,
        train_sizes = train_sizes_pct,
        cv          = cv_lc,
        scoring     = 'roc_auc',        # Métrique : AUC
        n_jobs      = -1,
        shuffle     = True,
        random_state= RANDOM_STATE
    )

    # Moyenne et écart-type sur les 5 folds
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    # Tracé des courbes
    ax.plot(train_sizes, train_mean, 'o-',
            color=color, label='Score Train', linewidth=2, markersize=5)
    ax.fill_between(train_sizes,
                    train_mean - train_std, train_mean + train_std,
                    alpha=0.15, color=color)

    ax.plot(train_sizes, val_mean, 's--',
            color='gray', label='Score Validation', linewidth=2, markersize=5)
    ax.fill_between(train_sizes,
                    val_mean - val_std, val_mean + val_std,
                    alpha=0.10, color='gray')

    # Ligne de référence baseline (random = 0.5)
    ax.axhline(0.5, color='red', linestyle=':', alpha=0.5, label='Baseline (random)')

    # Annotation du gap final
    gap = train_mean[-1] - val_mean[-1]
    ax.text(0.03, 0.05,
            f"Gap final: {gap:.3f}\n{'→ OK' if gap < 0.05 else '→ Overfitting'}",
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

    ax.set_title(f"{name}")
    ax.set_xlabel("Taille du set d'entraînement (lignes)")
    ax.set_ylabel("AUC (ROC)")
    ax.set_ylim(0.45, 0.85)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
plt.savefig('reports/01_learning_curves.png',
            dpi=130, bbox_inches='tight')
plt.close()
print("     ✓ Sauvegardé : 01_learning_curves.png")


# ──────────────────────────────────────────────────────────────────────────────
# GRAPHE 2 — MATRICES DE CONFUSION
# ──────────────────────────────────────────────────────────────────────────────
# Lecture de la matrice :
#   TP (haut-droite)  : a_corriger correctement détectés
#   FN (haut-gauche)  : a_corriger manqués (erreurs graves)
#   FP (bas-droite)   : validées incorrectement bloquées
#   TN (bas-gauche)   : validées correctement identifiées
# ──────────────────────────────────────────────────────────────────────────────

print("  → Graphe 2 : Matrices de Confusion")

fig2, axes = plt.subplots(2, 2, figsize=(12, 10))
fig2.suptitle(
    "Matrices de Confusion — Set de Test\n"
    "(Seuils optimisés pour Recall ≥ 0.72 sur la classe 'a_corriger')",
    fontsize=13, fontweight='bold'
)

labels = ['validee\n(0)', 'a_corriger\n(1)']

for ax, (name, res) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])

    # Normalisation par ligne pour afficher les pourcentages
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    # Heatmap avec annotations doubles (count + %)
    sns.heatmap(
        cm, ax=ax,
        annot=False,    # On fait les annotations manuellement
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        linewidths=0.5,
        linecolor='white',
        cbar=False
    )

    # Annotations personnalisées : count (grand) + % (petit)
    for i in range(2):
        for j in range(2):
            count = cm[i, j]
            pct   = cm_pct[i, j]
            # Couleur du texte selon la valeur de la case
            txt_color = 'white' if pct > 50 else 'black'
            ax.text(j + 0.5, i + 0.40,
                    f'{count}', ha='center', va='center',
                    fontsize=18, fontweight='bold', color=txt_color)
            ax.text(j + 0.5, i + 0.65,
                    f'({pct:.1f}%)', ha='center', va='center',
                    fontsize=10, color=txt_color)

    # Étiquettes des quadrants
    ax.text(-0.35, 0.5, 'Réel', va='center', ha='center',
            rotation=90, fontsize=10, fontweight='bold',
            transform=ax.transAxes)
    ax.text(0.5, -0.12, 'Prédit', va='center', ha='center',
            fontsize=10, fontweight='bold', transform=ax.transAxes)

    tp = results[name]['TP']
    fp = results[name]['FP']
    fn = results[name]['FN']
    tn = results[name]['TN']

    ax.set_title(
        f"{name}  (seuil={results[name]['threshold']:.2f})\n"
        f"AUC={results[name]['AUC']:.3f}  "
        f"Recall={results[name]['Recall']:.3f}  "
        f"F1={results[name]['F1']:.3f}",
        fontsize=10
    )
    ax.set_xlabel('Prédit', fontsize=9)
    ax.set_ylabel('Réel', fontsize=9)

plt.tight_layout()
plt.savefig('reports/02_confusion_matrices.png',
            dpi=130, bbox_inches='tight')
plt.close()
print("     ✓ Sauvegardé : 02_confusion_matrices.png")


# ──────────────────────────────────────────────────────────────────────────────
# GRAPHE 3 — COURBES ROC ET PRECISION-RECALL
# ──────────────────────────────────────────────────────────────────────────────

print("  → Graphe 3 : Courbes ROC et Precision-Recall")

fig3, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))
fig3.suptitle("Courbes ROC et Precision-Recall — Set de Test",
              fontsize=13, fontweight='bold')

# ── Courbe ROC ────────────────────────────────────────────────────────────────
for name, y_proba in probas_test.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = results[name]['AUC']
    t   = results[name]['threshold']

    ax_roc.plot(fpr, tpr,
                color=MODEL_COLORS[name], linewidth=2.5,
                label=f"{name} (AUC={auc:.3f})")

    # Marquer le point correspondant au seuil optimal
    pred_at_t = (y_proba >= t).astype(int)
    fpr_t = (((pred_at_t == 1) & (y_test == 0)).sum() /
              (y_test == 0).sum())
    tpr_t = results[name]['Recall']
    ax_roc.scatter(fpr_t, tpr_t,
                   color=MODEL_COLORS[name], s=120,
                   marker='D', zorder=5, edgecolors='black', linewidths=0.8)

ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Baseline (aléatoire)')
ax_roc.fill_between([0, 1], [0, 1], [0, 0], alpha=0.03, color='gray')
ax_roc.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=11)
ax_roc.set_ylabel('Taux de Vrais Positifs (TPR / Recall)', fontsize=11)
ax_roc.set_title('Courbe ROC\n(◆ = point au seuil optimal)')
ax_roc.legend(fontsize=9, loc='lower right')
ax_roc.set_xlim([-0.02, 1.02])
ax_roc.set_ylim([-0.02, 1.02])

# ── Courbe Précision-Rappel ───────────────────────────────────────────────────
# Plus informative que ROC sur des datasets déséquilibrés
baseline_pr = y_test.mean()   # Precision d'un classifieur naïf = proportion classe +

for name, y_proba in probas_test.items():
    precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_proba)

    ax_pr.plot(recall_arr, precision_arr,
               color=MODEL_COLORS[name], linewidth=2.5, label=name)

    # Point du seuil optimal
    ax_pr.scatter(results[name]['Recall'], results[name]['Precision'],
                  color=MODEL_COLORS[name], s=120,
                  marker='D', zorder=5, edgecolors='black', linewidths=0.8)

ax_pr.axhline(baseline_pr, color='red', linestyle='--', alpha=0.5,
              label=f'Baseline ({baseline_pr:.2f})')
ax_pr.axvline(0.72, color='green', linestyle=':', alpha=0.6,
              label='Seuil recall cible (0.72)')
ax_pr.set_xlabel('Recall (a_corriger)', fontsize=11)
ax_pr.set_ylabel('Precision (a_corriger)', fontsize=11)
ax_pr.set_title('Courbe Précision-Rappel\n(◆ = point au seuil optimal)')
ax_pr.legend(fontsize=9)
ax_pr.set_xlim([-0.02, 1.05])
ax_pr.set_ylim([0.40, 1.02])

plt.tight_layout()
plt.savefig('reports/03_roc_pr_curves.png',
            dpi=130, bbox_inches='tight')
plt.close()
print("     ✓ Sauvegardé : 03_roc_pr_curves.png")


# ──────────────────────────────────────────────────────────────────────────────
# GRAPHE 4 — COURBES SEUIL vs MÉTRIQUES
# ──────────────────────────────────────────────────────────────────────────────
# Montre comment Recall, Precision et F1 évoluent en fonction du seuil.
# Aide à comprendre et justifier le seuil choisi.
# ──────────────────────────────────────────────────────────────────────────────

print("  → Graphe 4 : Courbes Seuil → Métriques")

fig4, axes = plt.subplots(2, 2, figsize=(14, 10))
fig4.suptitle(
    "Impact du Seuil de Décision sur les Métriques\n"
    "(Calibré sur set de validation — ▼ = seuil retenu)",
    fontsize=13, fontweight='bold'
)

for ax, (name, model) in zip(axes.flatten(), models.items()):
    color = MODEL_COLORS[name]
    y_proba_val = model.predict_proba(X_val)[:, 1]

    # Recalculer les courbes sur le set de validation
    thresholds = np.arange(0.20, 0.80, 0.01)
    recalls, precisions, f1s = [], [], []

    for t in thresholds:
        p = (y_proba_val >= t).astype(int)
        recalls.append(recall_score(y_val, p, zero_division=0))
        precisions.append(precision_score(y_val, p, zero_division=0))
        f1s.append(f1_score(y_val, p, zero_division=0))

    ax.plot(thresholds, recalls,    color='#E53935', linewidth=2,
            label='Recall', linestyle='-')
    ax.plot(thresholds, precisions, color='#1E88E5', linewidth=2,
            label='Precision', linestyle='--')
    ax.plot(thresholds, f1s,        color='#43A047', linewidth=2.5,
            label='F1-Score', linestyle='-.')

    # Seuil retenu
    best_t = optimal_thrs[name]
    ax.axvline(best_t, color='black', linestyle=':', linewidth=2, alpha=0.8)
    best_f1_val = f1s[int((best_t - 0.20) / 0.01)]
    ax.annotate(f'  seuil={best_t:.2f}\n  F1={best_f1_val:.3f}',
                xy=(best_t, best_f1_val),
                fontsize=9, color='black',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.9))

    # Zone cible recall [0.72, 0.80]
    ax.axhspan(0.72, 0.80, alpha=0.08, color='green',
               label='Zone cible recall')
    ax.axhline(0.72, color='green', linestyle=':', alpha=0.4, linewidth=1)

    ax.set_title(f"{name}")
    ax.set_xlabel("Seuil de décision")
    ax.set_ylabel("Score")
    ax.set_ylim([0, 1.05])
    ax.legend(fontsize=8, loc='center left')
    ax.set_facecolor('#fafafa')

plt.tight_layout()
plt.savefig('reports/04_threshold_curves.png',
            dpi=130, bbox_inches='tight')
plt.close()
print("     ✓ Sauvegardé : 04_threshold_curves.png")


# ──────────────────────────────────────────────────────────────────────────────
# GRAPHE 5 — COMPARAISON INTER-MODÈLES (Tableau + Radar + Barres)
# ──────────────────────────────────────────────────────────────────────────────

print("  → Graphe 5 : Comparaison des modèles")

# ── 5a. Tableau de synthèse imprimé ──────────────────────────────────────────
print("\n" + "=" * 80)
print("  TABLEAU DE COMPARAISON DES MODÈLES — SET DE TEST")
print("=" * 80)

metrics_order = ['threshold', 'AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'Specificity']
header = f"  {'Modèle':<22}" + "".join(f"{m:>12}" for m in metrics_order)
print(header)
print("  " + "─" * 78)

for name, res in results.items():
    row = f"  {name:<22}"
    for m in metrics_order:
        row += f"{res[m]:>12.4f}"
    print(row)

print("  " + "─" * 78)
# Ligne des meilleures valeurs
best_per_metric = {}
for m in metrics_order[1:]:   # Skip threshold
    best_model = max(results, key=lambda n: results[n][m])
    best_per_metric[m] = (best_model, results[best_model][m])

best_row = f"  {'MEILLEUR':<22}"
best_row += f"{'—':>12}"   # threshold
for m in metrics_order[1:]:
    best_row += f"{best_per_metric[m][1]:>12.4f}"
print(best_row)
print("=" * 80)

# Afficher quel modèle est meilleur sur chaque métrique
print("\n  Meilleur modèle par métrique :")
for m, (model_name, val) in best_per_metric.items():
    print(f"    {m:<15} → {model_name:<22} ({val:.4f})")

# ── 5b. Graphe de comparaison ─────────────────────────────────────────────────
fig5 = plt.figure(figsize=(16, 12))
gs   = gridspec.GridSpec(2, 2, figure=fig5, hspace=0.4, wspace=0.35)

model_names = list(results.keys())
x_pos       = np.arange(len(model_names))
bar_colors  = [MODEL_COLORS[n] for n in model_names]

# ── Sous-graphe A : Barres groupées AUC / Recall / F1 ────────────────────────
ax_bar = fig5.add_subplot(gs[0, :])   # Toute la largeur

metrics_to_plot = ['AUC', 'Recall', 'F1', 'Precision', 'Accuracy']
bar_width = 0.15
offsets   = np.linspace(-(len(metrics_to_plot)-1)/2,
                         (len(metrics_to_plot)-1)/2,
                         len(metrics_to_plot)) * bar_width

metric_colors = ['#2196F3', '#E53935', '#43A047', '#FF9800', '#9C27B0']

for i, (metric, mc) in enumerate(zip(metrics_to_plot, metric_colors)):
    vals = [results[n][metric] for n in model_names]
    bars = ax_bar.bar(x_pos + offsets[i], vals,
                      width=bar_width, label=metric,
                      color=mc, alpha=0.85, edgecolor='white', linewidth=0.5)
    # Annotations sur les barres
    for bar, val in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom',
                    fontsize=7, rotation=45)

# Lignes de cible
ax_bar.axhline(0.72, color='red', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible Recall (0.72)')
ax_bar.axhline(0.68, color='green', linestyle='--', alpha=0.4,
               linewidth=1.2, label='Cible F1 (0.68)')

ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(model_names, fontsize=11)
ax_bar.set_ylabel('Score', fontsize=11)
ax_bar.set_title('Comparaison des Métriques par Modèle — Set de Test',
                  fontsize=12, fontweight='bold')
ax_bar.set_ylim(0, 1.15)
ax_bar.legend(fontsize=9, loc='upper right', ncol=4)
ax_bar.set_facecolor('#fafafa')

# Coloration des labels x selon le modèle
for tick, name in zip(ax_bar.get_xticklabels(), model_names):
    tick.set_color(MODEL_COLORS[name])
    tick.set_fontweight('bold')

# ── Sous-graphe B : TP/FP/FN/TN empilés ─────────────────────────────────────
ax_cm = fig5.add_subplot(gs[1, 0])

tp_vals = [results[n]['TP'] for n in model_names]
fp_vals = [results[n]['FP'] for n in model_names]
fn_vals = [results[n]['FN'] for n in model_names]
tn_vals = [results[n]['TN'] for n in model_names]

bw = 0.5
ax_cm.bar(x_pos, tp_vals, bw, label='TP (a_corriger détectés)',
          color='#43A047', alpha=0.9)
ax_cm.bar(x_pos, fn_vals, bw, bottom=tp_vals,
          label='FN (a_corriger manqués)', color='#E53935', alpha=0.9)
ax_cm.bar(x_pos, tn_vals, bw,
          bottom=[tp_vals[i]+fn_vals[i] for i in range(len(model_names))],
          label='TN (validées OK)', color='#1E88E5', alpha=0.9)
ax_cm.bar(x_pos, fp_vals, bw,
          bottom=[tp_vals[i]+fn_vals[i]+tn_vals[i] for i in range(len(model_names))],
          label='FP (validées bloquées)', color='#FF9800', alpha=0.9)

ax_cm.set_xticks(x_pos)
ax_cm.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_cm.set_ylabel('Nombre de prédictions')
ax_cm.set_title('Décomposition TP/FN/TN/FP\n(Test set = 287 lignes)',
                 fontsize=11, fontweight='bold')
ax_cm.legend(fontsize=7, loc='upper right')
ax_cm.set_facecolor('#fafafa')

# Annotations sur chaque barre
for i, name in enumerate(model_names):
    ax_cm.text(i, tp_vals[i]/2,
               f"TP\n{tp_vals[i]}", ha='center', va='center',
               fontsize=8, color='white', fontweight='bold')
    ax_cm.text(i, tp_vals[i] + fn_vals[i]/2,
               f"FN\n{fn_vals[i]}", ha='center', va='center',
               fontsize=8, color='white', fontweight='bold')

# ── Sous-graphe C : Seuil retenu par modèle ──────────────────────────────────
ax_thr = fig5.add_subplot(gs[1, 1])

thr_vals = [results[n]['threshold'] for n in model_names]
bars     = ax_thr.bar(x_pos, thr_vals, 0.5,
                       color=bar_colors, alpha=0.85,
                       edgecolor='white', linewidth=0.8)

ax_thr.axhline(0.5, color='black', linestyle='--', alpha=0.4,
               label='Seuil par défaut (0.5)')

for bar, t in zip(bars, thr_vals):
    ax_thr.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{t:.2f}', ha='center', va='bottom',
                fontsize=12, fontweight='bold')

ax_thr.set_xticks(x_pos)
ax_thr.set_xticklabels(model_names, fontsize=9, rotation=10)
ax_thr.set_ylabel('Seuil de décision optimal')
ax_thr.set_title('Seuil Optimal par Modèle\n(calibré sur set de validation)',
                  fontsize=11, fontweight='bold')
ax_thr.set_ylim(0, 0.80)
ax_thr.legend(fontsize=9)
ax_thr.set_facecolor('#fafafa')

plt.suptitle('Synthèse Comparative — Pipeline ML Vérification Factures',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('reports/05_model_comparison.png',
            dpi=130, bbox_inches='tight')
plt.close()
print("     ✓ Sauvegardé : 05_model_comparison.png")


# ──────────────────────────────────────────────────────────────────────────────
# GRAPHE 6 — IMPORTANCE DES FEATURES (GradientBoosting + RandomForest)
# ──────────────────────────────────────────────────────────────────────────────

print("  → Graphe 6 : Importance des features")

fig6, axes = plt.subplots(1, 2, figsize=(16, 7))
fig6.suptitle("Importance des Features\n(Validation de la pertinence du Feature Engineering)",
              fontsize=13, fontweight='bold')

for ax, name in zip(axes, ['GradientBoosting', 'RandomForest']):
    clf = models[name].named_steps['clf']

    importances = clf.feature_importances_
    indices     = np.argsort(importances)[::-1]

    # Séparation visuelle features originales vs synthétiques
    colors_feat = []
    for feat in ALL_FEATURES:
        if feat in FEATURES_ORIGINALES:
            colors_feat.append('#2196F3')    # Bleu = originale
        else:
            colors_feat.append('#E91E63')    # Rose = synthétique

    # Trier par importance
    sorted_colors = [colors_feat[i] for i in indices]
    sorted_names  = [ALL_FEATURES[i] for i in indices]

    bars = ax.barh(range(len(ALL_FEATURES)), importances[indices],
                   color=sorted_colors, alpha=0.85,
                   edgecolor='white', linewidth=0.5)

    ax.set_yticks(range(len(ALL_FEATURES)))
    ax.set_yticklabels(sorted_names, fontsize=9)
    ax.set_xlabel("Importance (Gini)", fontsize=10)
    ax.set_title(f"{name}", fontsize=11)
    ax.invert_yaxis()
    ax.set_facecolor('#fafafa')

    # Légende manuelle
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2196F3', label='Feature originale'),
        Patch(facecolor='#E91E63', label='Feature synthétique'),
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

    # Annotation de la feature #1
    top_feat = sorted_names[0]
    top_imp  = importances[indices[0]]
    ax.text(top_imp + 0.002, 0,
            f' {top_imp:.3f}', va='center', fontsize=8, color='black')

plt.tight_layout()
plt.savefig('reports/06_feature_importance.png',
            dpi=130, bbox_inches='tight')
plt.close()
print("     ✓ Sauvegardé : 06_feature_importance.png")


# ──────────────────────────────────────────────────────────────────────────────
# 7. RAPPORT FINAL
# ──────────────────────────────────────────────────────────────────────────────

print("\n[ÉTAPE 7/7] Rapport de synthèse final")
print("\n" + "=" * 70)
print("  RAPPORT FINAL — RÉSULTATS PAR MODÈLE")
print("=" * 70)

# Cibles définies
CIBLE_AUC    = (0.68, 0.75)
CIBLE_RECALL = (0.72, 0.80)
CIBLE_F1     = (0.68, 0.75)

for name, res in results.items():
    print(f"\n  ┌─ {name} {'─'*(40-len(name))}")
    print(f"  │  Seuil optimal      : {res['threshold']:.2f}  "
          f"(calibré sur set de validation)")
    print(f"  │  AUC                : {res['AUC']:.4f}  "
          f"{'✓ CIBLE' if CIBLE_AUC[0] <= res['AUC'] <= CIBLE_AUC[1] else '↗ Au-delà cible' if res['AUC'] > CIBLE_AUC[1] else '✗ Sous cible'}")
    print(f"  │  Accuracy           : {res['Accuracy']:.4f}")
    print(f"  │  Precision          : {res['Precision']:.4f}")
    print(f"  │  Recall (a_corriger): {res['Recall']:.4f}  "
          f"{'✓ CIBLE' if CIBLE_RECALL[0] <= res['Recall'] <= CIBLE_RECALL[1] else '↗ Au-delà cible' if res['Recall'] > CIBLE_RECALL[1] else '✗ Sous cible'}")
    print(f"  │  F1 (a_corriger)    : {res['F1']:.4f}  "
          f"{'✓ CIBLE' if CIBLE_F1[0] <= res['F1'] <= CIBLE_F1[1] else '↗ Au-delà cible' if res['F1'] > CIBLE_F1[1] else '✗ Sous cible'}")
    print(f"  │  Specificity        : {res['Specificity']:.4f}")
    print(f"  │  Matrice confusion  : TP={res['TP']} FP={res['FP']} "
          f"FN={res['FN']} TN={res['TN']}")
    print(f"  └──────────────────────────────────────────────")

    # Classification report complet
    print(f"\n  Classification Report — {name} :")
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['validee', 'a_corriger'],
        digits=4
    ))

# ── Meilleur modèle global ────────────────────────────────────────────────────
# Critère de sélection : F1 × 0.4 + Recall × 0.4 + AUC × 0.2
# (Recall pondéré fortement car manquer une erreur est coûteux)
scores_combined = {
    name: res['F1'] * 0.4 + res['Recall'] * 0.4 + res['AUC'] * 0.2
    for name, res in results.items()
}
best_model = max(scores_combined, key=scores_combined.get)

print("\n" + "=" * 70)
print(f"  RECOMMANDATION FINALE")
print("=" * 70)
print(f"\n  Meilleur modèle global (score = 0.4×F1 + 0.4×Recall + 0.2×AUC) :")
for name, score in sorted(scores_combined.items(), key=lambda x: -x[1]):
    flag = "  ← RECOMMANDÉ" if name == best_model else ""
    print(f"    {name:<22} score={score:.4f}{flag}")

print(f"\n  ➤ Modèle recommandé    : {best_model}")
print(f"  ➤ Seuil de décision    : {results[best_model]['threshold']:.2f}")
print(f"  ➤ AUC                  : {results[best_model]['AUC']:.4f}")
print(f"  ➤ Recall (a_corriger)  : {results[best_model]['Recall']:.4f}")
print(f"  ➤ F1 (a_corriger)      : {results[best_model]['F1']:.4f}")

print("\n" + "=" * 70)
print("  NOTE IMPORTANTE SUR LE PLAFOND THÉORIQUE")
print("=" * 70)
print("""
  Le plafond absolu de ce dataset est 64.2% d'accuracy (Bayes Error Rate).
  Cela signifie que 90% des lignes ont des features identiques mais des labels
  différents — la décision réelle repose sur des données absentes du dataset
  (valeurs brutes : montants, quantités, dates réelles).

  Pour dépasser 80% d'accuracy, il faudra enrichir le dataset avec :
    1. Les valeurs numériques réelles (quantite_acte, cout_total, etc.)
    2. L'historique du prescripteur (taux d'erreur par id_prescripteur)
    3. Des features temporelles (délais entre dates, saisonnalité)

  Référence : Bayes Error Rate — Fukunaga (1990), Pattern Recognition, Ch.3
""")

print("=" * 70)
print("  FICHIERS GÉNÉRÉS :")
print("    reports/01_learning_curves.png")
print("    reports/02_confusion_matrices.png")
print("    reports/03_roc_pr_curves.png")
print("    reports/04_threshold_curves.png")
print("    reports/05_model_comparison.png")
print("    reports/06_feature_importance.png")
print("=" * 70)


   PIPELINE ML — VÉRIFICATION FACTURES MÉDICALES

[ÉTAPE 1/7] Chargement des données
  ➤ Lignes totales       : 1433
  ➤ Classe 'validee'     : 600 (41.9%)
  ➤ Classe 'a_corriger'  : 833 (58.1%)
  ➤ Déséquilibre         : ratio = 1.39
  ➤ Plafond théorique    : 64.2% (limité par le bruit irréductible des données)

[ÉTAPE 2/7] Feature Engineering — Création des scores synthétiques
  ➤ Features originales utiles : 6
  ➤ Features synthétiques      : 13
  ➤ Total features             : 19

  Corrélations features synthétiques vs target :
    score_incompletude             +0.0595  █
    score_invalidite               -0.0734  ██
    score_incoherences             +0.0567  █
    score_absence_valeurs          -0.2066  ██████
    score_global_anomalie          -0.0319  
    consult_presta_combo           +0.1551  ████
    has_incoherence                +0.0497  █
    examen_avec_anomalie           +0.1516  ████
    taux_incompletude              +0.0595  █
    profil_risque                  